# 01: GTFS exploratory checks (public RTA data)

**GTFS** is a standard set of linked tables describing a transit service. The ones we use:
- `routes`: one row per route (a route has a type: bus, metro, ferry...)
- `trips`: one row per scheduled run of a route
- `stops`: one row per stop, with coordinates
- `stop_times`: which stops each trip visits, in order
- `shapes`: the drawn path each trip follows

The checks below ask: do the tables join up, are the IDs unique, and where is everything?

In [2]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FEED = ROOT / "data" / "interim" / "gtfs" / "feed"
print("Reading from:", FEED)

Reading from: /Users/isaiasbiju/METEORIC-Road-Bus-Robustness/data/interim/gtfs/feed


In [3]:
def read(name, **kw):
    # dtype=str keeps IDs exactly as written (no accidental number conversion)
    return pd.read_csv(FEED / name, dtype=str, encoding="utf-8-sig", **kw)

routes = read("routes.txt")
trips = read("trips.txt")
stops = read("stops.txt")
shapes = read("shapes.txt")
calendar = read("calendar.txt")
stop_times = read("stop_times.txt", usecols=["trip_id", "stop_id", "stop_sequence"])

for name, df in [("routes", routes), ("trips", trips), ("stops", stops),
                 ("shapes", shapes), ("calendar", calendar), ("stop_times", stop_times)]:
    print(f"{name:11s} {len(df):>10,d} rows | columns: {list(df.columns)}")

routes             207 rows | columns: ['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type', 'route_color', 'route_text_color']
trips           67,573 rows | columns: ['route_id', 'service_id', 'trip_id', 'shape_id', 'trip_headsign', 'direction_id', 'block_id', 'trip_short_name']
stops            2,848 rows | columns: ['stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'zone_id', 'location_type', 'stop_code', 'stop_desc', 'stop_url', 'platform_code']
shapes         180,554 rows | columns: ['shape_id', 'shape_pt_lat', 'shape_pt_lon', 'shape_pt_sequence', 'shape_dist_traveled']
calendar            18 rows | columns: ['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']
stop_times   1,420,200 rows | columns: ['trip_id', 'stop_id', 'stop_sequence']


## Which routes are buses?
In GTFS, `route_type` 3 means bus. The feed also holds metro, tram and ferry routes, which we exclude.

In [4]:
ROUTE_TYPES = {"0": "tram", "1": "metro", "2": "rail", "3": "bus", "4": "ferry"}
print(routes["route_type"].map(ROUTE_TYPES).fillna("other").value_counts().to_string())

bus_routes = routes[routes["route_type"] == "3"]
bus_trips = trips[trips["route_id"].isin(bus_routes["route_id"])]
print("\nbus routes:", len(bus_routes), "| bus trips:", f"{len(bus_trips):,}")
print("distinct route_short_name among bus routes:", bus_routes["route_short_name"].nunique())

route_type
bus      188
ferry     15
metro      3
tram       1

bus routes: 188 | bus trips: 57,854
distinct route_short_name among bus routes: 188


## Do the tables join up?
A **key** is a column that identifies a row (like `route_id`). Keys must be unique, and every key used in one table must exist in the table it points to. Counts marked "want 0" should be zero.

In [5]:
checks = {
    "route_id is unique": routes["route_id"].is_unique,
    "trip_id is unique": trips["trip_id"].is_unique,
    "stop_id is unique": stops["stop_id"].is_unique,
    "trips with unknown route_id (want 0)": int((~trips["route_id"].isin(routes["route_id"])).sum()),
    "trips with unknown shape_id (want 0)": int((~trips["shape_id"].isin(shapes["shape_id"].unique())).sum()),
    "trips with unknown service_id (want 0)": int((~trips["service_id"].isin(calendar["service_id"])).sum()),
    "stop_times with unknown stop_id (want 0)": int((~stop_times["stop_id"].isin(stops["stop_id"])).sum()),
    "stop_times with unknown trip_id (want 0)": int((~stop_times["trip_id"].isin(trips["trip_id"])).sum()),
    "stops never used by any trip (want 0)": int((~stops["stop_id"].isin(set(stop_times["stop_id"]))).sum()),
    "bus routes with no trips (want 0)": len(set(bus_routes["route_id"]) - set(trips["route_id"])),
}
for label, value in checks.items():
    print(f"{label:45s} {value}")

route_id is unique                            True
trip_id is unique                             True
stop_id is unique                             True
trips with unknown route_id (want 0)          0
trips with unknown shape_id (want 0)          0
trips with unknown service_id (want 0)        0
stop_times with unknown stop_id (want 0)      0
stop_times with unknown trip_id (want 0)      0
stops never used by any trip (want 0)         0
bus routes with no trips (want 0)             0


## Where is everything, and when?
Coordinates are latitude and longitude in degrees (WGS 84). Dubai city sits near 25.2 N, 55.3 E, so a range far wider than that means the feed covers more than the emirate.

In [6]:
lat = stops["stop_lat"].astype(float)
lon = stops["stop_lon"].astype(float)
print(f"stops: {len(stops):,} | missing coordinates: {int((lat.isna() | lon.isna()).sum())}")
print(f"stop latitude  {lat.min():.4f} to {lat.max():.4f}")
print(f"stop longitude {lon.min():.4f} to {lon.max():.4f}")
print("distinct shapes:", shapes["shape_id"].nunique())

print("\nservice dates:")
print(calendar[["service_id", "start_date", "end_date"]].drop_duplicates(["start_date", "end_date"]).to_string(index=False))

print("\ndirection_id counts (0 and 1 are the two travel directions of a route):")
print(trips["direction_id"].value_counts().to_string())

stops: 2,848 | missing coordinates: 0
stop latitude  24.2191 to 25.4106
stop longitude 54.3758 to 56.2940
distinct shapes: 443

service dates:
service_id start_date end_date
        US   20260911 20261231

direction_id counts (0 and 1 are the two travel directions of a route):
direction_id
0    41763
1    25810
